In [1]:
import pandas as pd
import numpy as np

from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.rdChemReactions import ReactionFromSmarts

from sklearn.preprocessing import MultiLabelBinarizer

import warnings

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings('ignore', module='upsetplot')

In [2]:
data = pd.read_csv("all_chem_df.csv")

data["tags"] = data["tags"].fillna("").str.split()
data = data.drop(["image_name", "Col3"], axis=1)
data = data[sorted(data.columns)]

data.head()

,smiles,tags
0,CC(=O)NC1C(O)OC(CO)C(O)C1O,[dermatologic]
1,CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...,[antiinfective]
2,CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...,[antiinfective]
3,COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...,[antineoplastic]
4,CC(=O)N[C@@H](CS)C(=O)[O-],"[antiinfective, respiratorysystem]"


In [3]:
data = data[data["tags"].apply(len) == 1].copy()
data["tags"] = data["tags"].apply(lambda x: x[0])
data = data.reset_index(drop=True)

data.head()

,smiles,tags
0,CC(=O)NC1C(O)OC(CO)C(O)C1O,dermatologic
1,CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...,antiinfective
2,CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...,antiinfective
3,COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...,antineoplastic
4,CC(=O)OCC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C,cns


In [4]:
def sanitize_smiles(smiles):
    """
    Czyści i normalizuje ciąg SMILES: usuwa sole, neutralizuje ładunki 
    i weryfikuje poprawność chemiczną. Zwraca czysty SMILES lub None.
    """
    if not isinstance(smiles, str):
        return None
        
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
        
    try:
        # Krok A: Zostawiamy tylko główną cząsteczkę (usuwamy sole, wodę itp.)
        # np. z zapisu leku i soli zostawi tylko lek
        fragment_chooser = rdMolStandardize.LargestFragmentChooser()
        mol = fragment_chooser.choose(mol)
        
        # Krok B: Neutralizacja ładunków
        # np. zamieni [O-] na OH, [NH3+] na NH2
        uncharger = rdMolStandardize.Uncharger()
        mol = uncharger.uncharge(mol)
        
        # Krok C: Podstawowa sanityzacja RDKit (naprawa aromatów, walencyjności)
        Chem.SanitizeMol(mol)
        
        # Krok D: Zwracamy z powrotem jako czysty, kanoniczny ciąg SMILES
        return Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
        
    except Exception as e:
        return None

def generate_mol(smiles):
    """Tworzy obiekt RDKit Mol na podstawie ciągu SMILES."""
    if not isinstance(smiles, str):
        return None
    return Chem.MolFromSmiles(smiles)

def generate_fp(mol, radius=2, nBits=2048):
    """Tworzy fingerprint w postaci tablicy numpy z obiektu Mol."""
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
        return np.array(fp)
    else:
        return np.zeros(nBits)

In [5]:
data["smiles"] = data["smiles"].apply(sanitize_smiles)
data["mol"] = data["smiles"].apply(generate_mol)
data["fingerprint"] = data["mol"].apply(generate_fp)

In [6]:
data

,smiles,tags,mol,fingerprint
0,CC(=O)NC1C(O)OC(CO)C(O)C1O,dermatologic,<rdkit.Chem.rdchem.Mol object at 0x104645000>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...,antiinfective,<rdkit.Chem.rdchem.Mol object at 0x1391c9070>,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...,antiinfective,<rdkit.Chem.rdchem.Mol object at 0x1391c90e0>,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
3,COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...,antineoplastic,<rdkit.Chem.rdchem.Mol object at 0x1391c9150>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."
4,CC(=O)OCC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C,cns,<rdkit.Chem.rdchem.Mol object at 0x1391c91c0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
6981,OCCN(CCO)c1nc(N2CCCCC2)c2nc(N(CCO)CCO)nc(N3CCC...,hematologic,<rdkit.Chem.rdchem.Mol object at 0x13928d930>,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6982,C=CO,hematologic,<rdkit.Chem.rdchem.Mol object at 0x13928d9a0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6983,CC1(C)SC2[C@H](NC(=O)[C@H](N)c3ccccc3)C(=O)N2[...,antiinfective,<rdkit.Chem.rdchem.Mol object at 0x13928da10>,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6984,COCCCOc1ccnc(C[S@@](=O)c2nc3ccccc3[nH]2)c1C,gastrointestinal,<rdkit.Chem.rdchem.Mol object at 0x13928da80>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."


In [7]:
bioisosteric_reactions = {
    "Cl_to_F": "[Cl:1]>>[F:1]",
    "F_to_Cl": "[F:1]>>[Cl:1]",
    "OH_to_NH2": "[O;H1:1]>>[N;H2:1]",
    "NH2_to_OH": "[N;H2:1]>>[O;H1:1]",
    "COOH_to_CONH2": "[C:2](=[O:3])[O;H1:1]>>[C:2](=[O:3])[N;H2:1]",
    "CONH2_to_COOH": "[C:2](=[O:3])[N;H2:1]>>[C:2](=[O:3])[O;H1:1]"
}

reactions = {k: ReactionFromSmarts(v) for k, v in bioisosteric_reactions.items()}

In [8]:
# 1. Wyciągamy dane do szybkich tablic numpy
smiles_arr = data["smiles"].to_numpy() 
fp_arr = np.stack(data["fingerprint"].values) 

rows = []

# 2. Grupowanie po tagach (teraz tagi to czyste stringi)
for tag, positions in sorted(data.groupby("tags").indices.items()):
    fps = fp_arr[positions]
    n = len(fps)

    # Podobieństwo cząsteczki do samej siebie = 1.0
    sim_sums = np.ones(n, dtype=float)

    # 3. Obliczanie Tanimoto (macierzowo) dla klas, które mają więcej niż 1 lek
    if n > 1:
        for i in range(1, n):
            # Iloczyn skalarny (część wspólna bitów)
            intersection = np.dot(fps[:i], fps[i])
            # Suma bitów dla poszczególnych wektorów
            sum_A = np.sum(fps[:i], axis=1)
            sum_B = np.sum(fps[i])
            
            # Wzór na Tanimoto dla wektorów binarnych
            sims = intersection / (sum_A + sum_B - intersection)
            
            sim_sums[i] += sims.sum()
            sim_sums[:i] += sims

    # Średnie podobieństwo do reszty grupy
    mean_tanimoto = sim_sums / n
    
    # 4. Sortowanie malejąco i wybór max 5 indeksów
    top_idx = np.argsort(-mean_tanimoto)[:min(5, n)]

    # 5. Zapisanie wyników
    for local_idx in top_idx:
        pos = positions[local_idx]
        rows.append(
            {
                "tags": tag,
                "original_smiles": smiles_arr[pos],
                "mean_tanimoto_to_tag": mean_tanimoto[local_idx],
            }
        )

# Tworzymy tabelę podsumowującą z naszymi reprezentantami
top5_by_tag = pd.DataFrame(rows)
display(top5_by_tag)

,tags,original_smiles,mean_tanimoto_to_tag
0,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.146960
1,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.146960
2,antiinfective,CC(=O)OCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)[C@H](...,0.146150
3,antiinfective,CO/N=C(\C(=O)N[C@@H]1C(=O)N2C(C(=O)O)=C(COC(C)...,0.146028
4,antiinfective,CO/N=C(/C(=O)N[C@@H]1C(=O)N2C(C(=O)O)=C(COC(C)...,0.146028
5,antiinflammatory,CC(=O)OCC(=O)[C@]1(O)[C@@H](C)C[C@H]2[C@@H]3CC...,0.190402
6,antiinflammatory,CC(=O)OCC(=O)[C@@]1(O)[C@H](C)C[C@H]2C3CCC4=CC...,0.190402
7,antiinflammatory,CC(=O)OCC(=O)[C@@]1(O)[C@@H](C)C[C@H]2C3CCC4=C...,0.190402
8,antiinflammatory,CC(=O)OCC(=O)[C@@]1(O)[C@@H](C)C[C@H]2[C@@H]3C...,0.190402
9,antiinflammatory,CC(=O)OCC(=O)[C@@]1(O)[C@@H](C)CC2C3CCC4=CC(=O...,0.190402


In [9]:
reaction_results = []

for index, row in top5_by_tag.iterrows():
    tag = row["tags"]
    original_smiles = row["original_smiles"]
    mean_tanimoto = row["mean_tanimoto_to_tag"]
    
    # Tworzymy obiekt cząsteczki z oryginalnego SMILES
    mol = Chem.MolFromSmiles(original_smiles)
    if mol is None:
        continue

    # 3. Sprawdzanie każdej z 6 reakcji na danej cząsteczce
    for rxn_name, rxn in reactions.items():
        # Uruchamiamy reakcję (zwraca krotkę krotek z produktami)
        products = rxn.RunReactants((mol,))
        
        # Używamy zbioru (set), aby od razu odrzucać identyczne produkty,
        # jeśli reakcja zaszła w kilku symetrycznych miejscach
        unique_new_smiles = set()
        
        for product_tuple in products:
            product_mol = product_tuple[0]
            
            try:
                # 4. Sanityzacja i kanonizacja nowego produktu
                Chem.SanitizeMol(product_mol)
                new_smiles = Chem.MolToSmiles(product_mol, isomericSmiles=True, canonical=True)
                unique_new_smiles.add(new_smiles)
            except Exception:
                # Jeśli powstanie twór łamiący zasady walencyjności, ignorujemy go
                pass
                
        # 5. Zapisywanie poprawnych wyników
        for new_smiles in unique_new_smiles:
            # Upewniamy się, że cząsteczka faktycznie uległa zmianie
            if new_smiles != original_smiles:
                reaction_results.append({
                    "original_tag": tag,
                    "original_smiles": original_smiles,
                    "mean_tanimoto_to_tag": mean_tanimoto,
                    "reaction_type": rxn_name,
                    "new_smiles": new_smiles
                })

isosteric_reactions_df = pd.DataFrame(reaction_results)

display(isosteric_reactions_df)

,original_tag,original_smiles,mean_tanimoto_to_tag,reaction_type,new_smiles
0,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.14696,OH_to_NH2,COCC1=C(C(N)=O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...
1,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.14696,NH2_to_OH,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...
2,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.14696,COOH_to_CONH2,COCC1=C(C(N)=O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...
3,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.14696,OH_to_NH2,COCC1=C(C(N)=O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...
4,antiinfective,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,0.14696,NH2_to_OH,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...
...,...,...,...,...,...
121,respiratorysystem,CC(C)[N+]1(C)C2CCC1CC(OC(=O)C(CO)c1ccccc1)C2,0.22130,OH_to_NH2,CC(C)[N+]1(C)C2CCC1CC(OC(=O)C(CN)c1ccccc1)C2
122,respiratorysystem,CC(C)[N+]1(C)[C@@H]2CC[C@H]1CC(OC(=O)C(CO)c1cc...,0.22130,OH_to_NH2,CC(C)[N+]1(C)[C@@H]2CC[C@H]1CC(OC(=O)C(CN)c1cc...
123,respiratorysystem,CC(C)[N+]1(C)C2CC[C@@H]1CC(OC(=O)C(CO)c1ccccc1)C2,0.22130,OH_to_NH2,CC(C)[N+]1(C)C2CC[C@@H]1CC(OC(=O)C(CN)c1ccccc1)C2
124,respiratorysystem,CC(C)[N+]1(C)[C@@H]2CC[C@H]1CC(OC(=O)[C@H](CO)...,0.22130,OH_to_NH2,CC(C)[N+]1(C)[C@@H]2CC[C@H]1CC(OC(=O)[C@H](CN)...


In [15]:
isosteres_data = pd.DataFrame({
    "original_smiles": isosteric_reactions_df["original_smiles"],
    "isostere_smiles": isosteric_reactions_df["new_smiles"]
})

isosteres_data = isosteres_data.drop_duplicates(subset=["isostere_smiles"]).reset_index(drop=True)
isosteres_data["mol"] = isosteres_data["isostere_smiles"].apply(generate_mol)
isosteres_data["fingerprint"] = isosteres_data["mol"].apply(generate_fp)

isosteres_data

,original_smiles,isostere_smiles,mol,fingerprint
0,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,COCC1=C(C(N)=O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,<rdkit.Chem.rdchem.Mol object at 0x14ee13a00>,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,COCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)/C(=N\OC)c3...,<rdkit.Chem.rdchem.Mol object at 0x14ee13f40>,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CC(=O)OCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)[C@H](...,CC(=O)OCC1=C(C(N)=O)N2C(=O)[C@@H](NC(=O)[C@H](...,<rdkit.Chem.rdchem.Mol object at 0x14ee13220>,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CC(=O)OCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)[C@H](...,CC(=O)OCC1=C(C(=O)O)N2C(=O)[C@@H](NC(=O)[C@H](...,<rdkit.Chem.rdchem.Mol object at 0x14ee137d0>,"[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CO/N=C(\C(=O)N[C@@H]1C(=O)N2C(C(=O)O)=C(COC(C)...,CO/N=C(\C(=O)N[C@@H]1C(=O)N2C(C(N)=O)=C(COC(C)...,<rdkit.Chem.rdchem.Mol object at 0x14ee138b0>,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
100,C#CC1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@@H]4[C@...,C#CC1(N)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@@H]4[C@...,<rdkit.Chem.rdchem.Mol object at 0x14ee7ea40>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
101,CC(C)[N+]1(C)C2CCC1CC(OC(=O)C(CO)c1ccccc1)C2,CC(C)[N+]1(C)C2CCC1CC(OC(=O)C(CN)c1ccccc1)C2,<rdkit.Chem.rdchem.Mol object at 0x14ee7eab0>,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
102,CC(C)[N+]1(C)[C@@H]2CC[C@H]1CC(OC(=O)C(CO)c1cc...,CC(C)[N+]1(C)[C@@H]2CC[C@H]1CC(OC(=O)C(CN)c1cc...,<rdkit.Chem.rdchem.Mol object at 0x14ee7eb20>,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
103,CC(C)[N+]1(C)C2CC[C@@H]1CC(OC(=O)C(CO)c1ccccc1)C2,CC(C)[N+]1(C)C2CC[C@@H]1CC(OC(=O)C(CN)c1ccccc1)C2,<rdkit.Chem.rdchem.Mol object at 0x14ee7eb90>,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
